🔗 [Back to Table of Contents](https://github.com/najaeda/najaeda-tutorials#-table-of-contents)

# Chapter 4: Loading SystemVerilog and Browsing the Elaborated Netlist

In this chapter, we'll load a small **SystemVerilog** design with `najaeda`, inspect the elaborated hierarchy, and look at the optional elaborated AST dump generated during loading.

---

## Setting Up the Environment

We'll start by installing **najaeda** and writing the SystemVerilog design to a local file.

In [ ]:
!pip install najaeda

In [ ]:
%%writefile systemverilog_stream_meter.sv
module sample_gate #(parameter int W = 8) (
  input logic [W-1:0] sample,
  input logic [W-1:0] mask,
  output logic [W-1:0] gated
);
  assign gated = sample & mask;
endmodule

module stream_meter #(parameter int W = 8, parameter int COUNT_W = 4) (
  input logic clk,
  input logic reset,
  input logic valid,
  input logic [W-1:0] sample,
  input logic [W-1:0] mask,
  output logic [W-1:0] sum,
  output logic [COUNT_W-1:0] valid_count,
  output logic alarm
);
  logic [W-1:0] gated_sample;
  logic [W-1:0] sum_next;

  sample_gate #(.W(W)) u_gate (
    .sample(sample),
    .mask(mask),
    .gated(gated_sample)
  );

  assign sum_next = sum + gated_sample;
  assign alarm = (valid_count == 4'd7);

  always_ff @(posedge clk)
    if (reset) begin
      sum <= '0;
      valid_count <= '0;
    end else if (valid) begin
      sum <= sum_next;
      valid_count <= valid_count + 'd1;
    end
endmodule

module top (
  input logic clk,
  input logic reset,
  input logic valid,
  input logic [7:0] sample,
  input logic [7:0] mask,
  output logic [7:0] sum,
  output logic [3:0] valid_count,
  output logic alarm
);
  stream_meter #(.W(8), .COUNT_W(4)) u_meter (
    .clk(clk),
    .reset(reset),
    .valid(valid),
    .sample(sample),
    .mask(mask),
    .sum(sum),
    .valid_count(valid_count),
    .alarm(alarm)
  );
endmodule


## Loading the SystemVerilog Design

We ask `najaeda` to load the `.sv` file and, at the same time, emit two optional artifacts:

- an elaborated AST JSON file
- a diagnostics report

Once loading is done, the returned `top` object is a regular `najaeda` instance that can be browsed just like a Verilog design.

In [ ]:
from collections import Counter
from pathlib import Path
import json

from IPython.display import Image, display
from najaeda import netlist

def term_summary(instance):
    return [
        (term.get_name(), term.get_width() if term.is_bus() else 1)
        for term in instance.get_terms()
    ]

def child_summary(instance):
    rows = []
    for index, child in enumerate(instance.get_child_instances(), start=1):
        name = child.get_name() or f"<unnamed_{index}>"
        model = child.get_model_name() or "<internal>"
        rows.append((name, model))
    return rows

def named_nets(instance):
    return [net.get_name() for net in instance.get_nets() if net.get_name()]

def display_dot(instance, stem):
    instance.dump_full_dot(f"{stem}.dot")
    !dot -Tpng {stem}.dot -o {stem}.png
    display(Image(filename=f"{stem}.png"))

netlist.reset()

ast_path = "systemverilog_stream_meter_elaborated_ast.json"
diagnostics_path = "systemverilog_stream_meter_diagnostics.txt"

top = netlist.load_system_verilog(
    "systemverilog_stream_meter.sv",
    config=netlist.SystemVerilogConfig(
        elaborated_ast_json_path=ast_path,
        diagnostics_report_path=diagnostics_path,
    ),
)

print(f"Loaded top instance: {top.get_name()} (model: {top.get_model_name()})")
print(f"Terms: {top.count_terms()} | child instances: {top.count_child_instances()}")

## Browsing the Top-Level Netlist

In [ ]:
print("Top-level terms (name, width):")
for term_name, width in term_summary(top):
    print(f"  {term_name}: {width}")

print("\nTop-level child instances:")
for name, model in child_summary(top):
    print(f"  {name}: {model}")

## Inspecting the Elaborated Hierarchy

The top instance contains one parameterized child named `u_meter`. Inside that instance, we still keep the named helper `u_gate`, while the accumulator, counter, compare, and sequential control are elaborated into primitive cells.

In [ ]:
u_meter = top.get_child_instance("u_meter")
u_gate = u_meter.get_child_instance("u_gate")

print(f"u_meter model: {u_meter.get_model_name()}")
print("u_meter terms (name, width):")
for term_name, width in term_summary(u_meter):
    print(f"  {term_name}: {width}")

print("\nu_meter named nets:")
for net_name in named_nets(u_meter):
    print(f"  {net_name}")

print("\nChild instance breakdown inside u_meter:")
for model_name, count in Counter(
    (child.get_model_name() or "<internal>") for child in u_meter.get_child_instances()
).most_common():
    print(f"  {model_name}: {count}")

registers = [child for child in u_meter.get_child_instances() if child.get_model_name() == "naja_dff"]
full_adders = [child for child in u_meter.get_child_instances() if child.get_model_name() == "naja_fa"]

print(f"\nRegister primitives found: {len(registers)}")
print(f"Adder cells found: {len(full_adders)}")
print(f"u_gate output is connected to upper net: {u_gate.get_term('gated').get_upper_net().get_name()}")

first_register = registers[0]
print("\nOne register connectivity sample:")
for term in first_register.get_terms():
    upper_net = term.get_upper_net()
    print(f"  {term.get_name()} -> {upper_net.get_name() if upper_net else '<unconnected>'}")

This is a more realistic elaborated structure:

- `u_gate` remains a named hierarchical helper module.
- `naja_dff` cells implement the registered state for `sum` and `valid_count`.
- `naja_fa` cells implement arithmetic for the accumulator and counter update.
- `xnor_2` and `and_4` cells participate in the threshold compare that drives `alarm`.

That gives us a compact but representative netlist with hierarchy, arithmetic, control, and sequential state.

## Visualizing the Elaborated Netlist

In [ ]:
display_dot(top, "systemverilog_stream_meter_top")
display_dot(u_meter, "systemverilog_stream_meter_u_meter")

✅ This concludes **Chapter 4** of the **najaeda** tutorial.

You now have a complete flow for loading a `.sv` file, navigating the elaborated hierarchy with the standard `najaeda` browsing API, inspecting a more representative functional block, visualizing the resulting netlist, and exporting the elaborated AST for additional inspection.